# Visualize Diffleop SDF Outputs

This notebook displays generated Diffleop molecules from SDF files in 3D using `py3Dmol`. It is configured to inspect the locally copied RunPod output under `outputs/sampling_dec_001`.


In [1]:
from pathlib import Path
import sys

def find_diffleop_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "configs").is_dir() and (path / "scripts").is_dir() and (path / "data").is_dir():
            return path
    raise RuntimeError("Could not find the Diffleop root directory")

ROOT = find_diffleop_root()
sys.path.insert(0, str(ROOT))
print(ROOT)


/Volumes/SATECHI_DISK_Media/Projects/github/gpcr-leop/Diffleop


In [2]:
try:
    import py3Dmol
except ImportError as exc:
    raise ImportError("Install py3Dmol first: mamba run -n diffleop pip install py3Dmol") from exc

from IPython.display import HTML, Markdown, display
from rdkit import Chem
from rdkit.Chem import AllChem
print("py3Dmol, IPython, and RDKit are available")


py3Dmol, IPython, and RDKit are available


## Select an Output

The default `RUN_DIR` points to the sampled output copied from RunPod: `outputs/sampling_dec_001`. Completed demo cases are `0` through `4`; `5` is a partial sixth case from the interrupted run.


In [3]:
RUN_DIR = ROOT / "outputs" / "sampling_dec_001"
OUTPUT_ID = "0"
SAMPLE_INDEX = 0

sample_dir = RUN_DIR / "sdf" / OUTPUT_ID
retain_path = sample_dir / "smiles_retain.smi"

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run output directory was not found: {RUN_DIR}")
if not sample_dir.exists():
    available = sorted(p.name for p in (RUN_DIR / "sdf").iterdir() if p.is_dir())
    raise FileNotFoundError(f"Sample directory was not found: {sample_dir}. Available IDs: {available}")

sdf_candidates = sorted(sample_dir.glob("*.sdf"))
if not sdf_candidates:
    raise FileNotFoundError(f"No SDF files found in {sample_dir}")

sdf_path = sample_dir / f"{SAMPLE_INDEX}.sdf"
if not sdf_path.exists():
    sdf_path = sdf_candidates[0]

available_ids = sorted(p.name for p in (RUN_DIR / "sdf").iterdir() if p.is_dir())
print("run_dir:", RUN_DIR)
print("available_ids:", available_ids)
print("sample_dir:", sample_dir)
print("sdf_path:", sdf_path, sdf_path.exists())
print("retain_path:", retain_path, retain_path.exists())


run_dir: /Volumes/SATECHI_DISK_Media/Projects/github/gpcr-leop/Diffleop/outputs/sampling_dec_001
available_ids: ['0', '1', '2', '3', '4', '5']
sample_dir: /Volumes/SATECHI_DISK_Media/Projects/github/gpcr-leop/Diffleop/outputs/sampling_dec_001/sdf/0
sdf_path: /Volumes/SATECHI_DISK_Media/Projects/github/gpcr-leop/Diffleop/outputs/sampling_dec_001/sdf/0/0.sdf True
retain_path: /Volumes/SATECHI_DISK_Media/Projects/github/gpcr-leop/Diffleop/outputs/sampling_dec_001/sdf/0/smiles_retain.smi True


## Input Metadata

`smiles_retain.smi` records the retained scaffold, masked fragment, original ligand, source ligand SDF, and protein pocket PDB for each demo input.


## Helpers

These helpers collect generated SDF files, resolve optional raw input files, and build fallback reference ligands from the original ligand SMILES when the raw ligand SDF is not available locally.


In [4]:
METADATA_KEYS = ["retain_smi", "mask_smi", "original_ligand_smi", "ligand_file", "protein_pocket_file"]

def read_retain_metadata(path):
    lines = path.read_text().splitlines()
    return dict(zip(METADATA_KEYS, lines))

def generated_sdf_paths(run_dir=RUN_DIR, include_partial=True):
    paths = sorted((run_dir / "sdf").glob("*/*.sdf"), key=lambda p: (int(p.parent.name), int(p.stem)))
    if include_partial:
        return paths
    return [p for p in paths if p.parent.name in {"0", "1", "2", "3", "4"}]

def resolve_existing_path(rel_path, sample_dir=None):
    rel_path = Path(rel_path)
    candidates = [
        ROOT / "data" / "demo" / "dec" / rel_path,
        ROOT / "data" / rel_path,
        ROOT / rel_path,
    ]
    if sample_dir is not None:
        candidates.append(sample_dir / rel_path.name)
    return next((p for p in candidates if p.exists()), None)

def sdf_text_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    status = AllChem.EmbedMolecule(mol, randomSeed=2024)
    if status == 0:
        AllChem.UFFOptimizeMolecule(mol, maxIters=200)
    mol = Chem.RemoveHs(mol)
    return Chem.MolToMolBlock(mol)

def original_ligand_sdf_text(metadata, sample_dir):
    ligand_path = resolve_existing_path(metadata["ligand_file"], sample_dir)
    if ligand_path is not None:
        return ligand_path.read_text(), ligand_path
    return sdf_text_from_smiles(metadata["original_ligand_smi"]), None

def protein_pdb_text(metadata, sample_dir):
    protein_path = resolve_existing_path(metadata["protein_pocket_file"], sample_dir)
    if protein_path is None:
        return None, None
    return protein_path.read_text(), protein_path

def show_generated(sdf_path, width=700, height=480):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(sdf_path.read_text(), "sdf")
    view.setStyle({"model": 0}, {"stick": {"radius": 0.18, "colorscheme": "greenCarbon"}})
    view.zoomTo()
    return view

def show_comparison(sdf_path, width=820, height=560):
    sample_dir = sdf_path.parent
    metadata = read_retain_metadata(sample_dir / "smiles_retain.smi")
    generated_sdf = sdf_path.read_text()
    original_sdf, ligand_path = original_ligand_sdf_text(metadata, sample_dir)
    pdb, protein_path = protein_pdb_text(metadata, sample_dir)

    view = py3Dmol.view(width=width, height=height)
    model_idx = 0
    if pdb is not None:
        view.addModel(pdb, "pdb")
        view.setStyle({"model": model_idx}, {"cartoon": {"color": "lightgray"}})
        view.addSurface(py3Dmol.VDW, {"opacity": 0.18, "color": "white"}, {"model": model_idx})
        model_idx += 1
    if original_sdf is not None:
        view.addModel(original_sdf, "sdf")
        view.setStyle({"model": model_idx}, {"stick": {"radius": 0.14, "colorscheme": "cyanCarbon"}})
        model_idx += 1
    view.addModel(generated_sdf, "sdf")
    view.setStyle({"model": model_idx}, {"stick": {"radius": 0.20, "colorscheme": "greenCarbon"}})
    view.zoomTo({"model": model_idx})
    return view, metadata, ligand_path, protein_path


def show_original_ligand_3d(sample_dir, width=700, height=480):
    metadata = read_retain_metadata(sample_dir / "smiles_retain.smi")
    original_sdf, ligand_path = original_ligand_sdf_text(metadata, sample_dir)
    if original_sdf is None:
        raise ValueError(f"Could not build original ligand 3D structure for {sample_dir}")
    view = py3Dmol.view(width=width, height=height)
    view.addModel(original_sdf, "sdf")
    view.setStyle({"model": 0}, {"stick": {"radius": 0.18, "colorscheme": "cyanCarbon"}})
    view.zoomTo()
    return view, metadata, ligand_path


def show_generated_overlay(sample_dir, width=820, height=560):
    sdf_files = sorted(sample_dir.glob("*.sdf"), key=lambda p: int(p.stem))
    if not sdf_files:
        raise FileNotFoundError(f"No SDF files found in {sample_dir}")

    colors = ["greenCarbon", "magentaCarbon", "orangeCarbon", "cyanCarbon", "purpleCarbon"]
    view = py3Dmol.view(width=width, height=height)
    for model_idx, sdf_file in enumerate(sdf_files):
        view.addModel(sdf_file.read_text(), "sdf")
        view.setStyle(
            {"model": model_idx},
            {"stick": {"radius": 0.16 + 0.02 * (model_idx == 0), "colorscheme": colors[model_idx % len(colors)]}},
        )
    view.zoomTo()
    return view, sdf_files

def show_case_overlay_with_reference(sample_dir, width=900, height=620):
    metadata = read_retain_metadata(sample_dir / "smiles_retain.smi")
    original_sdf, ligand_path = original_ligand_sdf_text(metadata, sample_dir)
    pdb, protein_path = protein_pdb_text(metadata, sample_dir)
    sdf_files = sorted(sample_dir.glob("*.sdf"), key=lambda p: int(p.stem))

    colors = ["greenCarbon", "magentaCarbon", "orangeCarbon", "purpleCarbon"]
    view = py3Dmol.view(width=width, height=height)
    model_idx = 0
    if pdb is not None:
        view.addModel(pdb, "pdb")
        view.setStyle({"model": model_idx}, {"cartoon": {"color": "lightgray"}})
        view.addSurface(py3Dmol.VDW, {"opacity": 0.16, "color": "white"}, {"model": model_idx})
        model_idx += 1
    if original_sdf is not None:
        view.addModel(original_sdf, "sdf")
        view.setStyle({"model": model_idx}, {"stick": {"radius": 0.12, "colorscheme": "cyanCarbon"}})
        model_idx += 1
    first_generated_model = model_idx
    for offset, sdf_file in enumerate(sdf_files):
        view.addModel(sdf_file.read_text(), "sdf")
        view.setStyle(
            {"model": model_idx},
            {"stick": {"radius": 0.18, "colorscheme": colors[offset % len(colors)]}},
        )
        model_idx += 1
    view.zoomTo({"model": first_generated_model})
    return view, metadata, ligand_path, protein_path, sdf_files

all_sdfs = generated_sdf_paths(include_partial=True)
completed_sdfs = generated_sdf_paths(include_partial=False)
print(f"all generated SDF files: {len(all_sdfs)}")
print(f"completed-case SDF files: {len(completed_sdfs)}")
for p in all_sdfs:
    print(p.relative_to(ROOT))


all generated SDF files: 10
completed-case SDF files: 9
outputs/sampling_dec_001/sdf/0/0.sdf
outputs/sampling_dec_001/sdf/0/1.sdf
outputs/sampling_dec_001/sdf/1/0.sdf
outputs/sampling_dec_001/sdf/1/1.sdf
outputs/sampling_dec_001/sdf/2/1.sdf
outputs/sampling_dec_001/sdf/3/0.sdf
outputs/sampling_dec_001/sdf/3/1.sdf
outputs/sampling_dec_001/sdf/4/0.sdf
outputs/sampling_dec_001/sdf/4/1.sdf
outputs/sampling_dec_001/sdf/5/0.sdf


## All Generated Molecules


In [5]:
for sdf_file in all_sdfs:
    display(Markdown(f"### `{sdf_file.relative_to(ROOT)}`"))
    show_generated(sdf_file).show()


### `outputs/sampling_dec_001/sdf/0/0.sdf`

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### `outputs/sampling_dec_001/sdf/0/1.sdf`

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### `outputs/sampling_dec_001/sdf/1/0.sdf`

...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### `outputs/sampling_dec_001/sdf/4/1.sdf`

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### `outputs/sampling_dec_001/sdf/5/0.sdf`

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Compare Generated Molecules with Input Ligand and Pocket

Generated molecules are shown in green. The input ligand is shown in cyan when the raw SDF exists; otherwise the original ligand SMILES is embedded into an approximate 3D conformer and shown in cyan. The protein pocket is added in gray/transparent white only when the raw PDB exists locally.


In [6]:
for sdf_file in all_sdfs:
    view, metadata, ligand_path, protein_path = show_comparison(sdf_file)
    display(Markdown(f"### `{sdf_file.relative_to(ROOT)}`"))
    print("retain_smi:", metadata["retain_smi"])
    print("mask_smi:", metadata["mask_smi"])
    print("original_ligand_smi:", metadata["original_ligand_smi"])
    print("ligand source:", ligand_path if ligand_path else "SMILES fallback; raw ligand SDF not found")
    print("protein source:", protein_path if protein_path else "raw pocket PDB not found")
    view.show()


### `outputs/sampling_dec_001/sdf/0/0.sdf`

retain_smi: Cc1cnn(C[C@H]2CN[C@H](C)CN2CC(=O)N2CC(C)(C)c3cnc(C[*:1])cc32)c1
mask_smi: c1ccc([*:1])cc1
original_ligand_smi: Cc1cnn(C[C@H]2CN[C@H](C)CN2CC(=O)N2CC(C)(C)c3cnc(Cc4ccccc4)cc32)c1
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### `outputs/sampling_dec_001/sdf/0/1.sdf`

retain_smi: Cc1cnn(C[C@H]2CN[C@H](C)CN2CC(=O)N2CC(C)(C)c3cnc(C[*:1])cc32)c1
mask_smi: c1ccc([*:1])cc1
original_ligand_smi: Cc1cnn(C[C@H]2CN[C@H](C)CN2CC(=O)N2CC(C)(C)c3cnc(Cc4ccccc4)cc32)c1
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found


...

retain_smi: c1ccc(COc2ccc([*:1])cc2)cc1
mask_smi: OCCC[*:1]
original_ligand_smi: OCCCc1ccc(OCc2ccccc2)cc1
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### `outputs/sampling_dec_001/sdf/5/0.sdf`

retain_smi: COc1ccc(OCCCCC[*:1])cc1Cc1cnc2nc(N)nc(N)c2c1C
mask_smi: O=C(O)[*:1]
original_ligand_smi: COc1ccc(OCCCCCC(=O)O)cc1Cc1cnc2nc(N)nc(N)c2c1C
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Compare One Selected Case


In [7]:
selected_view, selected_metadata, selected_ligand_path, selected_protein_path = show_comparison(sdf_path)
print("selected generated SDF:", sdf_path.relative_to(ROOT))
print("ligand source:", selected_ligand_path if selected_ligand_path else "SMILES fallback; raw ligand SDF not found")
print("protein source:", selected_protein_path if selected_protein_path else "raw pocket PDB not found")
selected_view.show()


selected generated SDF: outputs/sampling_dec_001/sdf/0/0.sdf
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Original Input Ligands in 3D

This section displays the original input ligand for each case. If the raw ligand SDF is not available locally, the notebook builds an approximate 3D conformer from `original_ligand_smi`.


In [8]:
for sample_path in sorted((RUN_DIR / "sdf").iterdir(), key=lambda p: int(p.name)):
    if not sample_path.is_dir():
        continue
    view, metadata, ligand_path = show_original_ligand_3d(sample_path)
    display(Markdown(f"### original input ligand for case `{sample_path.name}`"))
    print("original_ligand_smi:", metadata["original_ligand_smi"])
    print("ligand source:", ligand_path if ligand_path else "SMILES fallback; raw ligand SDF not found")
    view.show()


### original input ligand for case `0`

original_ligand_smi: Cc1cnn(C[C@H]2CN[C@H](C)CN2CC(=O)N2CC(C)(C)c3cnc(Cc4ccccc4)cc32)c1
ligand source: SMILES fallback; raw ligand SDF not found


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### original input ligand for case `1`

original_ligand_smi: CC1=C(COc2cccc([C@H](O)CCN)c2)C(C)(C)CCC1
ligand source: SMILES fallback; raw ligand SDF not found


...

original_ligand_smi: OCCCc1ccc(OCc2ccccc2)cc1
ligand source: SMILES fallback; raw ligand SDF not found


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### original input ligand for case `5`

original_ligand_smi: COc1ccc(OCCCCCC(=O)O)cc1Cc1cnc2nc(N)nc(N)c2c1C
ligand source: SMILES fallback; raw ligand SDF not found


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Overlay Generated Samples from the Same Input

`0.sdf` and `1.sdf` are two independently sampled outputs for the same input case because `sampling_dec.yml` uses `num_samples: 2`. This view overlays those generated samples case by case.


In [9]:
for sample_path in sorted((RUN_DIR / "sdf").iterdir(), key=lambda p: int(p.name)):
    if not sample_path.is_dir():
        continue
    sdf_files = sorted(sample_path.glob("*.sdf"), key=lambda p: int(p.stem))
    if len(sdf_files) < 2:
        print(f"skip {sample_path.name}: only {len(sdf_files)} generated SDF file(s)")
        continue
    view, files = show_generated_overlay(sample_path)
    display(Markdown(f"### case `{sample_path.name}`: " + ", ".join(f"`{p.name}`" for p in files)))
    print("green = 0.sdf, magenta = 1.sdf")
    view.show()


### case `0`: `0.sdf`, `1.sdf`

green = 0.sdf, magenta = 1.sdf


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### case `1`: `0.sdf`, `1.sdf`

green = 0.sdf, magenta = 1.sdf


...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### case `4`: `0.sdf`, `1.sdf`

green = 0.sdf, magenta = 1.sdf


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

skip 5: only 1 generated SDF file(s)


## Overlay Generated Samples with Input Ligand and Pocket

This view overlays all generated samples for each input case with the input ligand reference. The protein pocket is included only when the raw PDB exists locally.


In [10]:
for sample_path in sorted((RUN_DIR / "sdf").iterdir(), key=lambda p: int(p.name)):
    if not sample_path.is_dir():
        continue
    sdf_files = sorted(sample_path.glob("*.sdf"), key=lambda p: int(p.stem))
    if not sdf_files:
        continue
    view, metadata, ligand_path, protein_path, files = show_case_overlay_with_reference(sample_path)
    display(Markdown(f"### case `{sample_path.name}`"))
    print("cyan = input ligand/reference; green = 0.sdf; magenta = 1.sdf")
    print("original_ligand_smi:", metadata["original_ligand_smi"])
    print("ligand source:", ligand_path if ligand_path else "SMILES fallback; raw ligand SDF not found")
    print("protein source:", protein_path if protein_path else "raw pocket PDB not found")
    print("generated:", ", ".join(p.name for p in files))
    view.show()


### case `0`

cyan = input ligand/reference; green = 0.sdf; magenta = 1.sdf
original_ligand_smi: Cc1cnn(C[C@H]2CN[C@H](C)CN2CC(=O)N2CC(C)(C)c3cnc(Cc4ccccc4)cc32)c1
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found
generated: 0.sdf, 1.sdf


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### case `1`

cyan = input ligand/reference; green = 0.sdf; magenta = 1.sdf
original_ligand_smi: CC1=C(COc2cccc([C@H](O)CCN)c2)C(C)(C)CCC1
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found
generated: 0.sdf, 1.sdf


...

cyan = input ligand/reference; green = 0.sdf; magenta = 1.sdf
original_ligand_smi: OCCCc1ccc(OCc2ccccc2)cc1
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found
generated: 0.sdf, 1.sdf


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### case `5`

cyan = input ligand/reference; green = 0.sdf; magenta = 1.sdf
original_ligand_smi: COc1ccc(OCCCCCC(=O)O)cc1Cc1cnc2nc(N)nc(N)c2c1C
ligand source: SMILES fallback; raw ligand SDF not found
protein source: raw pocket PDB not found
generated: 0.sdf


3Dmol.js failed to load for some reason. Please check your browser console for error messages.